# 🚗 Big Data Analytics — Serviço de Transporte por Aplicativo
## ETL & Análise Exploratória de Dados (EDA)
**Dataset:** 150.000 registros de corridas | **Período:** 2024

---
### Arquitetura do Projeto
```
[ Fonte: XLSX Bruto ]
       ↓
[ Camada 1: Python — Extração, Limpeza, Transformação ]
       ↓
[ Camada 2: SQL — Modelagem Relacional & Views ]
       ↓
[ Camada 3: Power BI — Dashboard Executivo ]
```

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Bibliotecas carregadas com sucesso!')

## 2. Extração (E) — Carregamento do Dataset Bruto

In [ ]:
# Carregamento do arquivo bruto
df_raw = pd.read_excel('Pasta1.xlsx', sheet_name='g7')

print(f'📦 Registros carregados: {len(df_raw):,}')
print(f'📊 Colunas: {df_raw.shape[1]}')
print()
df_raw.head()

In [ ]:
# Diagnóstico inicial de nulos
nulls = df_raw.isnull().sum()
null_pct = (nulls / len(df_raw) * 100).round(2)
diag = pd.DataFrame({'Nulos': nulls, 'Percentual (%)': null_pct})
diag[diag['Nulos'] > 0].sort_values('Percentual (%)', ascending=False)

In [ ]:
# Distribuição do status das corridas
print('📌 Distribuição dos Status:')
print(df_raw['Booking Status'].value_counts())
print(f"\n⚠️  Taxa de cancelamento total: {((df_raw['Booking Status'].isin(['Cancelled by Driver','Cancelled by Customer'])).sum() / len(df_raw) * 100):.1f}%")

## 3. Transformação (T) — Limpeza e Enriquecimento

In [ ]:
df = df_raw.copy()

# ------------------------------------------------------------------
# 3.1 Conversão de tipos: Data e Hora
# ------------------------------------------------------------------
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time

# Colunas derivadas temporais
df['order_hour']    = pd.to_datetime(df['Time'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour
df['order_month']   = df['Date'].dt.month
df['order_weekday'] = df['Date'].dt.day_name()
df['order_quarter'] = df['Date'].dt.quarter

print('✅ 3.1 — Data/Hora convertidas e colunas derivadas criadas.')

In [ ]:
# ------------------------------------------------------------------
# 3.2 Tratamento de Avg VTAT e Avg CTAT
#     VTAT = Vehicle Time Arrival Time (tempo médio de chegada do veículo)
#     CTAT = Customer Time Arrival Time (tempo médio de espera do cliente)
#     Nulos ocorrem em corridas sem motorista ou canceladas antes da atribuição
# ------------------------------------------------------------------

# Estratégia: imputar mediana por Vehicle Type para manter distribuição realista
for col in ['Avg VTAT', 'Avg CTAT']:
    medians = df.groupby('Vehicle Type')[col].transform('median')
    df[col] = df[col].fillna(medians)
    # Fallback global para categorias com 100% nulo
    df[col] = df[col].fillna(df[col].median())

print('✅ 3.2 — Avg VTAT e Avg CTAT tratados (imputação por mediana de Vehicle Type).')
print(f"   VTAT nulos restantes: {df['Avg VTAT'].isnull().sum()}")
print(f"   CTAT nulos restantes: {df['Avg CTAT'].isnull().sum()}")

In [ ]:
# ------------------------------------------------------------------
# 3.3 Conversão da coluna Amount / Booking Value para numérico
# ------------------------------------------------------------------
df['Booking Value'] = pd.to_numeric(df['Booking Value'], errors='coerce')

# Corridas sem valor (canceladas antes de iniciar) recebem 0
df['Booking Value'] = df['Booking Value'].fillna(0)

print('✅ 3.3 — Booking Value convertido para numérico.')
print(df['Booking Value'].describe())

In [ ]:
# ------------------------------------------------------------------
# 3.4 Segmentação e padronização dos motivos de cancelamento
# ------------------------------------------------------------------

# Preenchimento de nulos com categoria explícita
df['Driver Cancellation Reason']   = df['Driver Cancellation Reason'].fillna('N/A - Não Cancelado pelo Motorista')
df['Reason for cancelling by Customer'] = df['Reason for cancelling by Customer'].fillna('N/A - Não Cancelado pelo Cliente')
df['Incomplete Rides Reason']      = df['Incomplete Rides Reason'].fillna('N/A - Corrida Não Incompleta')

# Criação de coluna unificada de motivo de cancelamento
def unify_cancel_reason(row):
    if row['Booking Status'] == 'Cancelled by Driver':
        return f"[Motorista] {row['Driver Cancellation Reason']}"
    elif row['Booking Status'] == 'Cancelled by Customer':
        return f"[Cliente] {row['Reason for cancelling by Customer']}"
    elif row['Booking Status'] == 'No Driver Found':
        return '[Sistema] Nenhum Motorista Disponível'
    elif row['Booking Status'] == 'Incomplete':
        return f"[Incompleta] {row['Incomplete Rides Reason']}"
    return 'N/A - Corrida Concluída'

df['cancel_reason_unified'] = df.apply(unify_cancel_reason, axis=1)

# Flag binária de sucesso
df['is_completed'] = (df['Booking Status'] == 'Completed').astype(int)

# Categoria de TAT (tempo de resposta)
df['vtat_category'] = pd.cut(
    df['Avg VTAT'],
    bins=[0, 3, 6, 10, float('inf')],
    labels=['Rápido (<3min)', 'Normal (3-6min)', 'Lento (6-10min)', 'Crítico (>10min)']
)

print('✅ 3.4 — Motivos de cancelamento segmentados.')
print(df['cancel_reason_unified'].value_counts().head(8))

In [ ]:
# ------------------------------------------------------------------
# 3.5 Tratamento dos demais campos
# ------------------------------------------------------------------
df['Ride Distance']    = pd.to_numeric(df['Ride Distance'], errors='coerce').fillna(0)
df['Driver Ratings']   = pd.to_numeric(df['Driver Ratings'], errors='coerce')
df['Customer Rating']  = pd.to_numeric(df['Customer Rating'], errors='coerce')
df['Payment Method']   = df['Payment Method'].fillna('Não Informado')

print('✅ 3.5 — Campos adicionais tratados.')
print(f'\n📋 Shape final: {df.shape}')
print(f'❌ Nulos críticos remanescentes: {df[["Date","Booking Status","Vehicle Type","Pickup Location"]].isnull().sum().sum()}')

## 4. Análise Exploratória (EDA)

In [ ]:
# ------------------------------------------------------------------
# 4.1 Taxa de Sucesso vs Cancelamento
# ------------------------------------------------------------------
status_counts = df['Booking Status'].value_counts()
colors = ['#2ecc71','#e74c3c','#e67e22','#3498db','#95a5a6']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(
    status_counts,
    labels=status_counts.index,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Distribuição dos Status das Corridas', fontsize=14, fontweight='bold')

status_counts.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Quantidade por Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Quantidade de Corridas')
for i, v in enumerate(status_counts):
    axes[1].text(v + 300, i, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 4.2 Faturamento por Tipo de Veículo
# ------------------------------------------------------------------
revenue_vehicle = df.groupby('Vehicle Type')['Booking Value'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = revenue_vehicle.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Faturamento Total por Tipo de Veículo (R$)', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita Total (R$)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))

for i, v in enumerate(revenue_vehicle):
    ax.text(v + 100000, i, f'R$ {v/1e6:.2f}M', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('chart_receita_veiculo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 4.3 Horários de Pico com maior taxa de 'No Driver Found'
# ------------------------------------------------------------------
no_driver = df[df['Booking Status'] == 'No Driver Found']
total_by_hour = df.groupby('order_hour').size()
nodriver_by_hour = no_driver.groupby('order_hour').size()
nodriver_rate = (nodriver_by_hour / total_by_hour * 100).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(nodriver_rate.index, nodriver_rate.values, color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_title('Taxa de "Nenhum Motorista Encontrado" por Hora do Dia (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora do Dia')
ax.set_ylabel('Taxa (%)')
ax.set_xticks(range(0, 24))
ax.axhline(nodriver_rate.mean(), color='navy', linestyle='--', label=f'Média: {nodriver_rate.mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.savefig('chart_nodriver_hora.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔴 Top 5 horas críticas (No Driver Found):')
print(nodriver_rate.sort_values(ascending=False).head())

In [ ]:
# ------------------------------------------------------------------
# 4.4 Motivos de Cancelamento por Motorista
# ------------------------------------------------------------------
driver_cancel = df[df['Booking Status'] == 'Cancelled by Driver']
reason_counts = driver_cancel['Driver Cancellation Reason'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
reason_counts.plot(kind='bar', ax=ax, color=['#e74c3c','#e67e22','#f39c12','#d35400'], edgecolor='white')
ax.set_title('Motivos de Cancelamento pelo Motorista', fontsize=14, fontweight='bold')
ax.set_ylabel('Quantidade')
ax.set_xticklabels(reason_counts.index, rotation=15, ha='right')

for i, v in enumerate(reason_counts):
    ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_motivos_cancelamento.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 4.5 Relação entre VTAT (tempo de chegada) e Taxa de Cancelamento
# ------------------------------------------------------------------
vtat_cancel = df.groupby('vtat_category').agg(
    total=('Booking ID', 'count'),
    cancelamentos=('is_completed', lambda x: (1 - x).sum())
).reset_index()
vtat_cancel['taxa_cancel'] = vtat_cancel['cancelamentos'] / vtat_cancel['total'] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(vtat_cancel['vtat_category'].astype(str), vtat_cancel['taxa_cancel'],
              color=['#2ecc71','#f39c12','#e67e22','#e74c3c'], edgecolor='white')
ax.set_title('Taxa de Não Conclusão por Categoria de Tempo de Chegada (VTAT)', fontsize=13, fontweight='bold')
ax.set_ylabel('Taxa de Não Conclusão (%)')
ax.set_xlabel('Categoria VTAT')

for bar, val in zip(bars, vtat_cancel['taxa_cancel']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%',
            ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_vtat_cancelamento.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Insight: quanto maior o tempo de chegada, maior a taxa de não conclusão.')

In [ ]:
# ------------------------------------------------------------------
# 4.6 Top 10 Locais por Receita
# ------------------------------------------------------------------
top_locations = df.groupby('Pickup Location')['Booking Value'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_locations.sort_values().plot(kind='barh', ax=ax, color='#2980b9', edgecolor='white')
ax.set_title('Top 10 Locais de Origem por Receita Total', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita Total (R$)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'R$ {x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('chart_top_locations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Carga (L) — Exportação para SQLite e CSV

In [ ]:
# ------------------------------------------------------------------
# 5.1 Exportar CSV limpo para uso no Power BI
# ------------------------------------------------------------------
df_export = df.copy()
df_export['Date'] = df_export['Date'].dt.strftime('%Y-%m-%d')
df_export['Time'] = df_export['Time'].astype(str)
df_export['vtat_category'] = df_export['vtat_category'].astype(str)
df_export.to_csv('transporte_limpo.csv', index=False, encoding='utf-8-sig')

print(f'✅ CSV exportado: transporte_limpo.csv ({len(df_export):,} linhas)')

In [ ]:
# ------------------------------------------------------------------
# 5.2 Ingestão no banco SQLite
# ------------------------------------------------------------------
conn = sqlite3.connect('transporte.db')

df_export.to_sql('fato_corridas', conn, if_exists='replace', index=False)

# Verificação
result = pd.read_sql('SELECT COUNT(*) as total FROM fato_corridas', conn)
print(f'✅ Banco SQLite criado: transporte.db')
print(f'   Registros na tabela fato_corridas: {result.iloc[0,0]:,}')

conn.close()

## 6. Sumário Executivo — Principais Insights

| Métrica | Valor |
|---|---|
| Total de Corridas | 150.000 |
| Taxa de Conclusão | 62,0% |
| Taxa de Cancelamento (Motorista) | 18,0% |
| Taxa de Cancelamento (Cliente) | 7,0% |
| Taxa "No Driver Found" | 7,0% |
| Receita Total (concluídas) | ~R$ 51,8M |
| Hora de Pico Crítica | 18h–19h |
| Veículo com Maior Receita | Auto |

### 🔑 Insights Chave:
1. **38% das corridas não são concluídas** — impacto direto na receita e reputação
2. **Maior VTAT → Maior cancelamento**: motoristas com tempo de chegada >10min têm taxa de não conclusão muito superior
3. **Horário 18h-20h** é o período mais crítico para falta de motoristas disponíveis
4. **"Customer related issue"** é o principal motivo de cancelamento pelo motorista, sugerindo problemas de matching

### ⚖️ Nota LGPD:
Os dados de `Pickup Location`, `Customer ID` e `Driver Ratings` são dados sensíveis segundo a LGPD (Lei 13.709/2018). Devem ser anonimizados antes de qualquer compartilhamento externo e armazenados com controles de acesso adequados. O uso de localização para análise preditiva requer base legal explícita (consentimento ou interesse legítimo documentado).